### Live Replay

In [1]:
import databento as db
from dotenv import load_dotenv
import pandas as pd
import os
load_dotenv()

with open("../../../universe.txt", 'r') as f:
    UNIVERSE = [line.strip() for line in f]
START = pd.Timestamp(2026, 6, 4, 9, 30, tz="US/Eastern")
END = pd.Timestamp(2026, 6, 4, 15, 50, tz="US/Eastern")

class Session_Volume_Accumulator:
    def __init__(self):
        self.counts = {}

    def counter(self, record: db.DBNRecord) -> None:
        if isinstance(record, db.MBP1Msg):
            id = record.instrument_id
            size = record.size
            if id not in self.counts.keys():
                self.counts[id] = size
            else:
                self.counts[id] += size
    
    def printer(self, record: db.DBNRecord) -> None:
        if isinstance(record, db.MBP1Msg):
            if record.size >= 500 and record.instrument_id == 33:
                print(record.size)
    
    def error_handler(self, exception: Exception) -> None:
        print(f"Exception: {exception}")

# Create a live client
client = db.Live(key=os.getenv("DATABENTO_API_KEY"))

# Subscribe with a specified start time for intraday replay
client.subscribe(
    dataset="EQUS.MINI",
    schema="tbbo",
    symbols=UNIVERSE,
    start=START,
)

acc = Session_Volume_Accumulator()
client.add_callback(record_callback=acc.counter,
                    exception_callback=acc.error_handler)
client.add_callback(record_callback=acc.printer)
client.start()
client.block_for_close(15)
client.stop()

600
585
1000
1000
570
677
700
566
1000
559
525
1000
700
1000
804
1000
1000
1000
839
649
639
543
846
500
1000
1000
565
955
600
1200
500
500
500
500
500
600
500
500
1152
840


In [2]:
acc.counts

{13340: 989127,
 16131: 1883514,
 20433: 274069,
 19729: 88634,
 11667: 6046797,
 10865: 2956031,
 13233: 581437,
 7151: 902904,
 10394: 439388,
 6297: 149302,
 17684: 1261278,
 16875: 68018,
 16244: 701633,
 1429: 2689323,
 262: 99982,
 33: 471176,
 16459: 88962,
 2994: 514385,
 5314: 299426,
 10924: 603409,
 17692: 1369319,
 773: 864486,
 18114: 18309,
 17200: 145436,
 2896: 151417,
 21095: 20672,
 58: 540145,
 13263: 402596,
 14261: 351405,
 10852: 247900,
 7756: 104212,
 14820: 479568,
 3356: 6088159,
 981: 166388,
 20790: 21156,
 2351: 120471,
 10345: 479783,
 13345: 112183,
 11872: 161390,
 20270: 35529,
 6823: 92332,
 3728: 277904,
 7152: 1559186,
 10888: 878315,
 1344: 25177,
 6905: 381868,
 16248: 1324900,
 10451: 783119,
 853: 1914759,
 5683: 12518,
 10724: 321421,
 8147: 4160156,
 8705: 1224719,
 17448: 1052062,
 38: 1311525,
 15002: 445036,
 2021: 393617,
 9976: 57402,
 8676: 125503,
 7716: 1071715,
 5960: 180102,
 3667: 326067,
 13760: 334716,
 10996: 1132899,
 3548: 35528

### Looking at data costs

In [ ]:
import databento as db
import pandas as pd
from dotenv import load_dotenv
load_dotenv()
import os
START = pd.Timestamp(2024,1,1,00,00,00)
END = pd.Timestamp(2026,7,2,00,00,00)
DATASET = "XNAS.ITCH"
SCHEMAS = ["imbalance", "tbbo", "statistics"]
# SCHEMAS = ["ohlcv-1d"]
# UNIVERSE = "ALL_SYMBOLS"
with open("../top_500_universe.txt", "r") as f:
    UNIVERSE = [line.strip() for line in f]


client = db.Historical(os.getenv("DATABENTO_API_KEY"))
total_cost = 0
total_size = 0

for schema in SCHEMAS:
    cost = client.metadata.get_cost(
        dataset=DATASET,
        symbols=UNIVERSE,
        start=START,
        end=END,
        schema=schema
    )

    size = client.metadata.get_billable_size(
        dataset=DATASET,
        symbols=UNIVERSE,
        start=START,
        end=END,
        schema=schema
    )
    print(f"{schema} Cost: ${cost:,.2f}, Uncompressed Size: {size / 1e9:,.2f} GB")
    total_cost += cost
    total_size += size

print(f"Total Cost: ${total_cost:,.2f}, Uncompressed Size: {total_size / 1e9:,.2f} GB")

imbalance Cost: $51.33, Uncompressed Size: 3.57 GB
tbbo Cost: $638.88, Uncompressed Size: 224.91 GB
statistics Cost: $0.00, Uncompressed Size: 0.01 GB
Total Cost: $690.21, Uncompressed Size: 228.49 GB


## Universe Selection

In [2]:
from data_collection.data_adapters import DatabentoGenericAdapter
from dotenv import load_dotenv
from pathlib import Path
import os
import polars as pl
load_dotenv()
pl.enable_string_cache()

CACHE_DIR = Path.cwd().resolve().parents[0] / "data" / "dbn_cache" / "generic"
adapter = DatabentoGenericAdapter(os.getenv("DATABENTO-API-KEY"), CACHE_DIR)

print(f"CACHE_DIR: {CACHE_DIR}")

START = "2025-01-01"
END = "2025-06-10"
DATASET = "XNAS.ITCH"

day_bars = (
    await adapter.get_data(
        tickers="ALL_SYMBOLS",
        start_date=START,
        end_date=END,
        dataset=DATASET,
        schema="ohlcv-1d"
    )
)

defs = (
    await adapter.get_data(
        tickers="ALL_SYMBOLS",
        start_date=START,
        end_date=END,
        dataset=DATASET,
        schema="definition"
    )
)

CACHE_DIR: /Users/michaelcusack-nelkin/GitHub/Summer_Research_2026/data/dbn_cache/generic
[GenericAdapter] Parquet cache hit: 071ad5a677ce9293c6d67d8690241537.parquet
[GenericAdapter] Parquet cache hit: dfe53a630b267abb902eaad84d85a1dd.parquet


In [43]:
nasdaq_stocks = (
    defs
    .group_by("symbol")
    .agg(
        pl.col("security_type").last().alias("sec_type"),
        (pl.col("exchange") == "XNAS").all().alias("xnas_full_window"),
        pl.col("exchange").n_unique().alias("n_exch"),
    )
    .filter(
        (pl.col("sec_type") == "C") & pl.col("xnas_full_window"),
        ~pl.col("symbol").str.contains(r"ZZT$|^Z[A-Z]YZ|^ZBZX|^ZSPC|^ZWZ")
    )
    .select("symbol")
    .sort("symbol")
)
print(f"{nasdaq_stocks.height} common stocks listed on XNAS for the full window")
nasdaq_stocks = nasdaq_stocks.to_series().to_list()
nasdaq_stocks

2700 common stocks listed on XNAS for the full window


['AADI',
 'AAL',
 'AAME',
 'AAOI',
 'AAON',
 'AAPL',
 'AARD',
 'ABAT',
 'ABEO',
 'ABL',
 'ABNB',
 'ABOS',
 'ABP',
 'ABSI',
 'ABUS',
 'ABVC',
 'ABVE',
 'ACAD',
 'ACB',
 'ACCD',
 'ACDC',
 'ACET',
 'ACGL',
 'ACHC',
 'ACHV',
 'ACIC',
 'ACIU',
 'ACIW',
 'ACLS',
 'ACLX',
 'ACMR',
 'ACNB',
 'ACNT',
 'ACOG',
 'ACON',
 'ACRS',
 'ACRV',
 'ACT',
 'ACTG',
 'ACTU',
 'ACXP',
 'ADBE',
 'ADEA',
 'ADGM',
 'ADI',
 'ADIL',
 'ADMA',
 'ADN',
 'ADP',
 'ADPT',
 'ADSK',
 'ADTN',
 'ADTX',
 'ADUR',
 'ADUS',
 'ADV',
 'ADVB',
 'ADVM',
 'AEHL',
 'AEHR',
 'AEI',
 'AEIS',
 'AEMD',
 'AENT',
 'AEP',
 'AEYE',
 'AFBI',
 'AFCG',
 'AFMD',
 'AFRM',
 'AFYA',
 'AGAE',
 'AGEN',
 'AGFY',
 'AGH',
 'AGIO',
 'AGNC',
 'AGRI',
 'AGYS',
 'AHCO',
 'AIEV',
 'AIFF',
 'AIHS',
 'AIMD',
 'AIOT',
 'AIP',
 'AIRE',
 'AIRG',
 'AIRJ',
 'AIRO',
 'AIRS',
 'AIRT',
 'AISP',
 'AKAM',
 'AKBA',
 'AKRO',
 'AKYA',
 'ALAB',
 'ALBT',
 'ALCE',
 'ALCO',
 'ALDX',
 'ALEC',
 'ALGM',
 'ALGN',
 'ALGS',
 'ALGT',
 'ALHC',
 'ALKT',
 'ALLK',
 'ALLO',
 'ALLR',
 'ALM

In [ ]:
k = 50
top_k = (
    day_bars
    .select(["ts_event", "symbol", "close", "volume"])
    .group_by(
        by="symbol"
    )
    .agg(
        (pl.col("close")*pl.col("volume")).mean().alias("ADV")
    )
    .rename({"by":"symbol"})
    .filter(pl.col("symbol").is_in(nasdaq_stocks))
    .top_k(k, by="ADV")
    .select("symbol")
    .to_series()
    .to_list()
)

# -- Write to Unvierse File --
with open(f"../top_{k}_universe.txt", "x") as f:
    f.write("\n".join(top_k))

In [45]:
top_k

['NVDA',
 'TSLA',
 'AAPL',
 'MSFT',
 'AMZN',
 'META',
 'GOOGL',
 'AVGO',
 'PLTR',
 'MSTR',
 'GOOG',
 'NFLX',
 'AMD',
 'COST',
 'COIN',
 'APP',
 'MU',
 'SMCI',
 'QCOM',
 'TXN',
 'CRWD',
 'CSCO',
 'INTC',
 'HOOD',
 'ADBE',
 'BKNG',
 'MRVL',
 'AMAT',
 'TMUS',
 'PEP',
 'INTU',
 'ISRG',
 'LRCX',
 'CRWV',
 'DASH',
 'GILD',
 'AMGN',
 'ADI',
 'KLAC',
 'PANW',
 'SBUX',
 'CMCSA',
 'CEG',
 'HON',
 'VRTX',
 'MELI',
 'NXPI',
 'PYPL',
 'ABNB',
 'REGN']

## Random

In [23]:
import numpy as np
lower = 1
upper = 500
arr = np.arange(lower, upper + 1)

arr

array([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
        14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,
        40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,
        66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,
        79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,
        92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104,
       105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117,
       118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130,
       131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143,
       144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156,
       157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169,
       170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 18

In [27]:
z = (arr - lower) / (upper - lower)
z = 2 * z - 1
z

array([-1.        , -0.99599198, -0.99198397, -0.98797595, -0.98396794,
       -0.97995992, -0.9759519 , -0.97194389, -0.96793587, -0.96392786,
       -0.95991984, -0.95591182, -0.95190381, -0.94789579, -0.94388778,
       -0.93987976, -0.93587174, -0.93186373, -0.92785571, -0.9238477 ,
       -0.91983968, -0.91583166, -0.91182365, -0.90781563, -0.90380762,
       -0.8997996 , -0.89579158, -0.89178357, -0.88777555, -0.88376754,
       -0.87975952, -0.8757515 , -0.87174349, -0.86773547, -0.86372745,
       -0.85971944, -0.85571142, -0.85170341, -0.84769539, -0.84368737,
       -0.83967936, -0.83567134, -0.83166333, -0.82765531, -0.82364729,
       -0.81963928, -0.81563126, -0.81162325, -0.80761523, -0.80360721,
       -0.7995992 , -0.79559118, -0.79158317, -0.78757515, -0.78356713,
       -0.77955912, -0.7755511 , -0.77154309, -0.76753507, -0.76352705,
       -0.75951904, -0.75551102, -0.75150301, -0.74749499, -0.74348697,
       -0.73947896, -0.73547094, -0.73146293, -0.72745491, -0.72

In [1]:
with open("../universe.txt", "r") as f:
    UNIVERSE = [line.strip() for line in f]

In [3]:
UNIVERSE[:50]

['NVDA',
 'SPY',
 'TSLA',
 'QQQ',
 'AAPL',
 'MSFT',
 'AMZN',
 'META',
 'GOOGL',
 'AVGO',
 'AMD',
 'TLT',
 'GOOG',
 'IWM',
 'NFLX',
 'TQQQ',
 'MU',
 'COST',
 'ADBE',
 'QCOM',
 'COIN',
 'SMCI',
 'INTC',
 'SMH',
 'MSTR',
 'TXN',
 'AMAT',
 'PEP',
 'CSCO',
 'BKNG',
 'ASML',
 'LRCX',
 'TMUS',
 'HYG',
 'INTU',
 'PDD',
 'PANW',
 'CRWD',
 'MRVL',
 'CMCSA',
 'ADI',
 'AMGN',
 'LIN',
 'VOO',
 'LQD',
 'SBUX',
 'KLAC',
 'PYPL',
 'HON',
 'IVV']